# FinAgent Colab Quick Start (L4 GPU)

This notebook uploads `finagent_colab.zip` to Colab, installs dependencies,
and runs the demo / retrieval benchmark / RAGAS evaluation.

**Runtime requirement:** use a GPU runtime — this pipeline is tuned for the
**L4 (24 GB)** instance (`Runtime → Change runtime type → L4`). The default
models are:

- Embedding: `intfloat/e5-mistral-7b-instruct` — 7B LLM encoder, ~14 GB
  download, runs in bf16 (~14 GB VRAM).
- Reranker: `BAAI/bge-reranker-v2-gemma` — 2.5B LLM cross-encoder, ~10 GB
  download, runs in fp16 (~5 GB VRAM).

The pipeline loads the two models **sequentially** (embed → offload to CPU →
rerank), so peak VRAM is ~14 GB and an L4 has plenty of headroom.

## Prepare the zip

In your local project root:

```bash
# Without Dataset (small)
bash scripts/export_colab_zip.sh

# With Dataset (~68 MB, no Kaggle download needed in Colab)
bash scripts/export_colab_zip.sh --with-data
```

If you skip the Dataset, the notebook will automatically download it from Kaggle
(requires uploading `access_token` or setting the `KAGGLE_API_TOKEN` Secret).

Run the first cell below to upload `finagent_colab.zip`.

In [ ]:
from google.colab import files
print('Please upload finagent_colab.zip')
uploaded = files.upload()

In [ ]:
import os, zipfile

os.makedirs('/content/finagent', exist_ok=True)
for fn in uploaded.keys():
    with zipfile.ZipFile(fn) as z:
        z.extractall('/content/finagent')

os.chdir('/content/finagent')
print('Working directory:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

In [ ]:
!pip install -q uv
!uv sync --python 3.12
print('Dependencies installed')

## GPU Setup (L4)

Install the CUDA build of torch into the project venv (the plain `uv sync`
above installs the CPU wheel) and verify the GPU is visible.

Note: if you re-run the `uv sync` cell later, re-run this cell afterwards —
`uv sync` would replace the CUDA wheel with the CPU wheel.

In [ ]:
!nvidia-smi
!uv pip install --python .venv/bin/python \
    --extra-index-url https://download.pytorch.org/whl/cu121 \
    "torch==2.2.2+cu121"
!.venv/bin/python -c "import torch; print('torch', torch.__version__); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"

import os
# Reduce CUDA fragmentation (inherited by the `uv run` subprocesses)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
assert torch.cuda.is_available(), 'No GPU runtime — set Runtime > Change runtime type > L4'
free_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU memory: {free_gb:.1f} GB')
assert free_gb > 14, f'Need a >=16GB GPU (L4 recommended) for the default models, got {free_gb:.1f} GB'

## Pre-download Models (~25 GB)

Downloads the embedding model, reranker, and the small RAGAS eval embeddings
into the Colab session cache up front, so progress is visible and later cells
don't stall on downloads.

In [ ]:
!.venv/bin/python - <<'PY'
from huggingface_hub import snapshot_download

jobs = [
    # 7B LLM embedding model (skip duplicate *.bin weights, keep safetensors)
    ("intfloat/e5-mistral-7b-instruct", ["*.bin"]),
    # 2.5B LLM reranker
    ("BAAI/bge-reranker-v2-gemma", None),
    # small encoder used by RAGAS answer-relevancy
    ("BAAI/bge-large-en-v1.5", None),
]
for repo, ignore in jobs:
    print("Downloading", repo, flush=True)
    snapshot_download(repo, ignore_patterns=ignore)
    print("  done:", repo, flush=True)
print("All models cached")
PY

## Download Kaggle Data (optional)

If the uploaded zip does not contain `Dataset/`, this cell will:
1. Read `KAGGLE_API_TOKEN` from Colab secrets, or prompt you to upload `access_token`
2. Install the Kaggle CLI
3. Download and extract the ICAIF-24 data into `Dataset/`

If you packaged with `--with-data`, this step is skipped automatically.

In [ ]:
import os, pathlib, subprocess

data_ok = pathlib.Path('Dataset/financebench_corpus.jsonl/corpus.jsonl').exists()
if data_ok:
    print('Dataset already present — skipping Kaggle download')
else:
    print('Dataset not found — downloading from Kaggle...')
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    token_path = os.path.expanduser('~/.kaggle/access_token')

    if not os.path.exists(token_path):
        try:
            from google.colab import userdata
            token = userdata.get('KAGGLE_API_TOKEN')
            if token:
                with open(token_path, 'w') as f:
                    f.write(token)
                os.chmod(token_path, 0o600)
                print('Loaded KAGGLE_API_TOKEN from Colab Secrets')
        except Exception:
            pass

    if not os.path.exists(token_path):
        print('Please upload your ~/.kaggle/access_token file')
        from google.colab import files
        uploaded_tok = files.upload()
        for fn in uploaded_tok:
            content = uploaded_tok[fn]
            mode = 'wb' if isinstance(content, bytes) else 'w'
            with open(token_path, mode) as f:
                f.write(content)
            os.chmod(token_path, 0o600)
            print(f'Saved {fn} -> {token_path}')

    subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
    os.makedirs('/tmp/kaggle_data', exist_ok=True)
    subprocess.run(
        ['kaggle', 'competitions', 'download',
         '-c', 'icaif-24-finance-rag-challenge', '-p', '/tmp/kaggle_data'],
        check=True,
    )
    subprocess.run(
        ['unzip', '-o',
         '/tmp/kaggle_data/icaif-24-finance-rag-challenge.zip', '-d', 'Dataset'],
        check=True,
    )
    print('Kaggle data downloaded and extracted')

## Configure API Keys (optional)

Only needed for MultiQuery expansion or RAGAS generation evaluation.

Recommended: add `DEEPSEEK_API_KEY` or `QWEN_API_KEY` via Colab Secrets (left panel).

In [ ]:
from google.colab import userdata
import os

for key in ['DEEPSEEK_API_KEY', 'QWEN_API_KEY', 'DEEPSEEK_MODEL', 'QWEN_MODEL']:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass

print('DEEPSEEK_API_KEY set:', bool(os.environ.get('DEEPSEEK_API_KEY')))
print('QWEN_API_KEY set:', bool(os.environ.get('QWEN_API_KEY')))

## Dependency-Free Demo

Verify the environment and pipeline flow without any models or API keys.

In [ ]:
!uv run python scripts/demo_synthetic.py

## Retrieval Benchmark

Uses `intfloat/e5-mistral-7b-instruct` (dense) + BM25 → RRF → MMR →
`BAAI/bge-reranker-v2-gemma` (LLM rerank).

Memory: the two LLM models are **never resident at the same time**. The
benchmark first runs the dense-retrieval pass with the embedding model
(~14 GB bf16), offloads it to CPU RAM, and only then loads the reranker
(~5 GB fp16). Peak VRAM ≈ 14 GB. If you ever hit CUDA OOM anyway, lower the
batch sizes:

```python
import os
os.environ['FINAGENT_EMBED_BATCH_SIZE'] = '2'
os.environ['FINAGENT_RERANK_BATCH_SIZE'] = '4'
```

In [ ]:
!uv run python main.py --dataset financebench --no-multiquery

## Generate Results Summary

After running all 7 datasets, this prints an NDCG@10 summary table
similar to the Kaggle reference notebook.

To reuse previously saved results:

```bash
uv run python scripts/summarize_results.py --from-json results/all.json
```

In [ ]:
# Run all 7 datasets and print a Markdown summary table (time-consuming)
!uv run python scripts/summarize_results.py --no-multiquery

## RAGAS Evaluation (requires API key)

Evaluates answer faithfulness, relevancy, and context utilization.

```bash
uv run python eval.py --config deepseek_baseline --dataset financebench
```

In [ ]:
!uv run python eval.py --config deepseek_baseline --dataset financebench --n 5